# 02 - Preprocessing & Feature Scaling

**Why this notebook exists:** in notebook 01 you called `KNeighborsClassifier` and `SVC`
without ever scaling the data. Those two models measure *distance* between points,
so a column measured in hundreds (`chol`) silently drowns out a column measured in 0/1 (`sex`).

By the end you will be able to:
1. Show, with numbers, that scaling changes KNN/SVM accuracy.
2. Pick between `StandardScaler`, `MinMaxScaler` and `RobustScaler`.
3. Encode categorical columns and fill missing values.
4. Wire it all into a single `ColumnTransformer`.

**Rule of thumb:** scale for KNN, SVM, Logistic Regression, PCA and neural nets.
Trees (Decision Tree, Random Forest, Gradient Boosting) do **not** need it.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
np.random.seed(42)

heart = pd.read_csv("../data/heart-disease.csv")
X = heart.drop("target", axis=1)
y = heart["target"]
print(X.shape, y.value_counts().to_dict())
heart.head()

## 1. The raw feature ranges

Look at how different the scales are. `chol` runs to roughly 560, `sex` is 0 or 1.

In [ ]:
X.describe().T[["min", "max", "mean", "std"]].round(2)

## 2. KNN without scaling vs with scaling

Same model, same split, same seed. The only difference is one `StandardScaler`.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

raw = KNeighborsClassifier().fit(X_train, y_train)
scaled = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier()),
]).fit(X_train, y_train)

print(f"KNN, raw features    : {raw.score(X_test, y_test):.3f}")
print(f"KNN, scaled features : {scaled.score(X_test, y_test):.3f}")

> **Read the gap.** That difference is not a better algorithm - it is the same algorithm
> finally able to see all the features. This is the most common silent bug for beginners.

Now the same test for SVM.

In [ ]:
from sklearn.svm import SVC

raw_svm = SVC().fit(X_train, y_train)
scaled_svm = Pipeline([("scaler", StandardScaler()), ("svm", SVC())]).fit(X_train, y_train)

print(f"SVM, raw features    : {raw_svm.score(X_test, y_test):.3f}")
print(f"SVM, scaled features : {scaled_svm.score(X_test, y_test):.3f}")

In [ ]:
# Control group: a tree, which is scale-invariant by design.
from sklearn.ensemble import RandomForestClassifier

raw_rf = RandomForestClassifier(random_state=42).fit(X_train, y_train)
scaled_rf = Pipeline([("scaler", StandardScaler()),
                      ("rf", RandomForestClassifier(random_state=42))]).fit(X_train, y_train)

print(f"Random Forest, raw    : {raw_rf.score(X_test, y_test):.3f}")
print(f"Random Forest, scaled : {scaled_rf.score(X_test, y_test):.3f}   <- basically unchanged")

## 3. Which scaler?

| Scaler | What it does | Use when |
|---|---|---|
| `StandardScaler` | mean 0, std 1 | default choice, roughly bell-shaped data |
| `MinMaxScaler` | squeezes into [0, 1] | you need a bounded range (e.g. image pixels) |
| `RobustScaler` | centres on median, scales by IQR | the column has outliers |

In [ ]:
from sklearn.preprocessing import MinMaxScaler, RobustScaler

col = X[["chol"]]
comparison = pd.DataFrame({
    "original":  col["chol"].values,
    "standard":  StandardScaler().fit_transform(col).ravel(),
    "minmax":    MinMaxScaler().fit_transform(col).ravel(),
    "robust":    RobustScaler().fit_transform(col).ravel(),
})
comparison.describe().T[["min", "max", "mean", "std"]].round(2)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3))
for ax, name in zip(axes, comparison.columns):
    ax.hist(comparison[name], bins=30, color="#4C78A8")
    ax.set_title(name)
plt.suptitle("Same column, four scalings - watch the x-axis, not the shape")
plt.tight_layout()
plt.show()

**Notice the shape never changes.** Scaling moves and stretches the axis; it does not
make a skewed column normal. That is a different tool (log transform, `PowerTransformer`).

## 4. Categorical columns and missing values

`heart-disease.csv` is all numeric. Switch to the car sales data, which has both text
columns and missing values - the realistic case.

In [ ]:
cars = pd.read_csv("../data/car-sales-extended-missing-data.csv")
print(cars.dtypes)
print("\nmissing values per column:")
print(cars.isna().sum())
cars.head()

In [ ]:
# Rows with no target are useless for supervised learning - drop them. Never impute y.
cars = cars.dropna(subset=["Price"])

X_cars = cars.drop("Price", axis=1)
y_cars = cars["Price"]

categorical = ["Make", "Colour"]
numeric = ["Odometer (KM)", "Doors"]
print(f"{len(X_cars)} rows kept")

### The `ColumnTransformer`

Different column types need different treatment. `ColumnTransformer` applies a
separate mini-pipeline to each group and glues the results back together.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor

categorical_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="constant", fill_value="missing")),
    ("encode", OneHotEncoder(handle_unknown="ignore")),
])

numeric_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

preprocessor = ColumnTransformer([
    ("cat", categorical_pipe, categorical),
    ("num", numeric_pipe, numeric),
])

model = Pipeline([
    ("prep", preprocessor),
    ("rf", RandomForestRegressor(random_state=42)),
])

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_cars, y_cars, test_size=0.2, random_state=42)

model.fit(Xc_train, yc_train)
print(f"R^2 on test set: {model.score(Xc_test, yc_test):.3f}")

In [ ]:
# What did the preprocessor actually produce?
transformed = preprocessor.fit_transform(Xc_train)
print(f"{Xc_train.shape[1]} original columns -> {transformed.shape[1]} numeric columns")
print(preprocessor.get_feature_names_out())

### Why `handle_unknown="ignore"` matters

If the test set contains a car colour that never appeared in training, a strict encoder
crashes at predict time. `"ignore"` encodes it as all zeros instead. In production that is
the difference between a working API and a 500 error.

---
## Exercises

1. Change the KNN comparison to use `MinMaxScaler` instead of `StandardScaler`.
   Does the conclusion change?
2. Set `n_neighbors=1` and re-run the raw-vs-scaled comparison. Does scaling matter
   *more* or *less* for a very flexible model? Explain why.
3. In the car sales pipeline, swap `SimpleImputer(strategy="median")` for `"mean"`.
   Look at the `Odometer (KM)` distribution first and predict which will do better.
4. Use `RobustScaler` in the numeric pipeline instead of `StandardScaler` and compare R^2.
5. **Stretch:** `Doors` is numeric but really categorical (3, 4, 5). Move it into the
   categorical group and see whether the score improves.